# ذكاء سوق العمل اليمني | دفتر Google Colab موحّد: **AraBERT + LSTM**

**مبني على مشروع `LMI_PROJECT` (نسخة V3) ومتوافق مع إضافة `LMI_GEO_NEURAL_V3_PATCH.zip`.**

هذا الدفتر يستدعي ملفات التدريب الحقيقية للمشروع، وليس محاكاة للتدريب:

- أربعة نماذج **LSTM**: المهارات، التخصصات، المسميات الوظيفية، المحافظات. تدريبها **مفعّل افتراضياً** ويعمل على CPU.
- ثلاثة نماذج **AraBERT**: تصنيف نوع الفرصة، النطاق الجغرافي، واستخراج الكيانات NER. التدريب مفعّل تلقائياً **عند وجود GPU**؛ يمكن فرضه على CPU ولكن سيكون بطيئاً جداً.
- **تقييم Test فعلي** مع مقارنة LSTM بـ Naive وMoving Average، وحفظ النماذج والنتائج في Google Drive.
- جميع المقاييس الخاصة بالنصوص **Silver** ما لم توجد تسميات بشرية Gold معتمدة. نتائج LSTM مبنية على حصص شهرية من بيانات المشروع.

> **مهم:** ملف `LMI_GEO_NEURAL_V3_PATCH.zip` مجرد **تحديث جغرافي** ولا يحتوي وحده على بيانات التدريب أو سكربت LSTM. تحتاج إلى **المشروع الكامل** في `MyDrive/LMI_PROJECT` أو ZIP المشروع الكامل مثل `LMI_PROJECT_V3_VERIFIED.zip`. يمكن تطبيق التحديث الجغرافي اختيارياً من الإعدادات أدناه، وهو **ليس** تدريب LSTM.

**ترتيب التشغيل:** الإعدادات ← ربط Drive ← تثبيت المكتبات ← تجهيز البيانات ← التدريب والتقييم ← الاختبارات ← التطبيق (اختياري). لا توجد مخرجات تدريب مسبقة مختلقة في هذا الدفتر؛ ستظهر المخرجات بعد التشغيل الفعلي في Colab.

**إصلاح الإصدار:** يفحص السكربت خيارات تدريب LSTM ويصلح عدم التوافق بين إصدارات المشروع قبل التنفيذ، مع نسخة احتياطية وسجل كامل للأخطاء.


## 1) إعدادات التشغيل — عدّل الخيارات هنا فقط عند الحاجة

In [ ]:
# الإعدادات الأساسية
PROJECT_DIR_NAME = 'LMI_PROJECT'                 # مجلد المشروع في MyDrive
PROJECT_ZIPS = ('LMI_PROJECT_V3_VERIFIED.zip', 'LMI_PROJECT_ENGINEERED.zip', 'LMI_PROJECT.zip')
GEO_PATCH_ZIP = 'LMI_GEO_NEURAL_V3_PATCH.zip'    # اختياري داخل MyDrive أو MyDrive/LMI_INPUT
APPLY_GEO_PATCH = False                         # True لتطبيق تحديث GeoScopeNetV2 على المشروع في Drive

# LSTM: 'force' يعيد التدريب فعلياً؛ 'missing' يتخطى الأوزان السليمة ويعيد تدريب الناقص.
LSTM_TRAIN_MODE = 'force'
LSTM_FAMILIES = ('skills', 'specializations', 'job_titles', 'governorates')
# تاريخ التقييم المرجعي لحماية تقسيم الزمن من أشهر لم تكتمل. None = تاريخ اليوم.
AS_OF = '2026-11-08'

# AraBERT: auto = عند وجود GPU فقط، true = حتى على CPU (بطيء)، false = عدم التدريب.
ARABERT_MODE = 'auto'
RETRAIN_ARABERT = False                           # True لإعادة تدريب النموذج حتى إذا كان محفوظاً
EVALUATE_ARABERT = True                           # تقييم Test للنماذج التي تتوفر أوزانها

# واجهة Streamlit تُشغَّل اختيارياً في آخر الدفتر.
RUN_STREAMLIT = False

assert LSTM_TRAIN_MODE in ('force', 'missing')
assert ARABERT_MODE in ('auto', 'true', 'false')
assert set(LSTM_FAMILIES).issubset({'skills','specializations','job_titles','governorates'})
print('LSTM mode:', LSTM_TRAIN_MODE, '| AraBERT:', ARABERT_MODE, '| Geographic patch:', APPLY_GEO_PATCH)

## 2) ربط Google Drive والعثور على المشروع الكامل (أو فك ZIP بأمان)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, date
from zoneinfo import ZoneInfo
import os, sys, json, shutil, zipfile, subprocess, time, tempfile, importlib.util

MY = Path('/content/drive/MyDrive')
PROJ = MY / PROJECT_DIR_NAME
WORK = Path('/content/lmi')
INPUT = MY / 'LMI_INPUT'

# لا نفك تحديث GeoScope الصغير باعتباره مشروعاً كاملاً.
def safe_extract_project(zip_path: Path, target: Path):
    with zipfile.ZipFile(zip_path) as z:
        members = z.infolist()
        files = [i for i in members if i.filename.startswith('LMI_PROJECT/')]
        if not files:
            raise ValueError(f'ليس مشروعاً كاملاً (لا يوجد LMI_PROJECT/): {zip_path.name}')
        for item in files:
            name = item.filename
            if '\\' in name or name.startswith('/') or '..' in Path(name).parts:
                raise ValueError(f'مسار ZIP غير آمن: {name}')
        for item in files:
            z.extract(item, path=target)

if not (PROJ/'src/forecasting/train_lstm.py').is_file():
    whole = next((p for p in [MY / name for name in PROJECT_ZIPS] + [INPUT / name for name in PROJECT_ZIPS] if p.is_file()), None)
    if whole is not None:
        print('استخراج المشروع الكامل:',whole)
        safe_extract_project(whole, MY)
    else:
        names = [f'LMI_{i:02d}_' for i in range(1,7)]
        parts = sorted([p for p in INPUT.glob('LMI_*.zip') if any(p.name.startswith(prefix) for prefix in names)])
        if len(parts) == 6 and all(any(p.name.startswith(prefix) for p in parts) for prefix in names):
            for p in parts:
                print('استخراج جزء:',p.name)
                safe_extract_project(p, MY)
        else:
            raise FileNotFoundError(
                'تحتاج إلى المشروع الكامل: MyDrive/LMI_PROJECT أو LMI_PROJECT_V3_VERIFIED.zip.\n'
                'ملف LMI_GEO_NEURAL_V3_PATCH.zip وحده تحديث فقط وليس مشروعاً للتدريب.'
            )

required = ('src/forecasting/train_lstm.py','src/forecasting/series.py',
            'config/training.json','FINAL_TRAINING_DATASET/unique_opportunities.csv',
            'FINAL_TRAINING_DATASET/skills/job_skills.csv')
missing = [s for s in required if not (PROJ/s).is_file()]
if missing:
    raise FileNotFoundError('المشروع غير مكتمل؛ ينقصه: '+', '.join(missing))

(PROJ/'models_dl').mkdir(exist_ok=True)
(PROJ/'reports').mkdir(exist_ok=True)
print('PROJECT OK:', PROJ)
print('LSTM training script:',PROJ/'src/forecasting/train_lstm.py')
print('Data:', (PROJ/'FINAL_TRAINING_DATASET/unique_opportunities.csv').stat().st_size//1024//1024,'MiB')

## 3) تثبيت متطلبات LSTM (والنماذج اللغوية عند الحاجة)

In [ ]:
# Colab يحتوي عادة على numpy/pandas/torch، لذلك لا نعيد تنزيلها إذا كانت موجودة.
import importlib.util, subprocess, sys
core = [('torch','torch>=2.1'), ('numpy','numpy>=1.24'), ('pandas','pandas>=2.0'),
        ('matplotlib','matplotlib'), ('sklearn','scikit-learn>=1.3'), ('pytest','pytest')]
missing_pkgs = [spec for mod,spec in core if importlib.util.find_spec(mod) is None]
if missing_pkgs:
    subprocess.run([sys.executable,'-m','pip','install',*missing_pkgs],check=True)
import torch, numpy as np, pandas as pd
print('Python',sys.version.split()[0],'PyTorch',torch.__version__,'Pandas',pd.__version__)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'غير متاح — تدريب LSTM يعمل على CPU')

TRAIN_ARABERT = (torch.cuda.is_available() if ARABERT_MODE=='auto' else ARABERT_MODE=='true')
HF_PATHS = (PROJ/'models_dl/classification/opportunity_type',
            PROJ/'models_dl/classification/geographic_scope', PROJ/'models_dl/ner')
HF_PRESENT = any(any((d/file).is_file() for file in
                 ('model.safetensors','pytorch_model.bin','model.safetensors.index.json','pytorch_model.bin.index.json'))
                 for d in HF_PATHS)
if TRAIN_ARABERT or (EVALUATE_ARABERT and HF_PRESENT):
    packages = [('transformers','transformers>=4.44,<5'),('accelerate','accelerate'),
                ('safetensors','safetensors'),('seqeval','seqeval'),('arabert','arabert')]
    pending=[spec for mod,spec in packages if importlib.util.find_spec(mod) is None]
    if pending:
        subprocess.run([sys.executable,'-m','pip','install',*pending],check=True)
    if importlib.util.find_spec('transformers'):
        import transformers
        if int(transformers.__version__.split('.')[0]) >= 5:
            raise RuntimeError('هذا المشروع يتطلب transformers<5؛ أعد تثبيته ثم Restart runtime')
if RUN_STREAMLIT and importlib.util.find_spec('streamlit') is None:
    subprocess.run([sys.executable,'-m','pip','install','streamlit>=1.40','plotly>=5.20'],check=True)
print('AraBERT training:', 'مفعّل' if TRAIN_ARABERT else 'غير مفعّل في هذا التشغيل (GPU غير متاح أو خيار الإعدادات)')

## 4) تحديث جغرافي اختياري — لا علاقة له بتدريب LSTM

In [ ]:
# يتوقف التحديث عند أي فشل ولا يطبع نجاحاً غير حقيقي. يحفظ المثبّت نسخاً احتياطية.
if APPLY_GEO_PATCH:
    patch = next((p for p in (MY/GEO_PATCH_ZIP, INPUT/GEO_PATCH_ZIP) if p.is_file()), None)
    if patch is None:
        raise FileNotFoundError(f'ضع {GEO_PATCH_ZIP} داخل MyDrive أو MyDrive/LMI_INPUT')
    patch_extract = Path('/content/lmi_geo_patch')
    if patch_extract.exists(): shutil.rmtree(patch_extract)
    with zipfile.ZipFile(patch) as z:
        for item in z.infolist():
            if item.filename.startswith('/') or '..' in Path(item.filename).parts or '\\' in item.filename:
                raise ValueError(f'Unsafe patch member: {item.filename}')
        z.extractall(patch_extract)
    installer = patch_extract/'LMI_GEO_NEURAL_V3_PATCH/apply_fix.py'
    if not installer.is_file(): raise FileNotFoundError('المثبت الأصلي غير موجود داخل التحديث')
    penv=os.environ.copy()
    penv.update(PYTHONUTF8='1', PYTHONIOENCODING='utf-8', PYTHONPATH=str(PROJ))
    subprocess.run([sys.executable,str(installer),str(PROJ)],check=True,cwd=installer.parent,env=penv)
    print('تم تطبيق تحديث GeoScopeNetV2 في Drive مع اختبار الاستدلال.')
else:
    print('لم تتغير ملفات التصنيف الجغرافي. يمكن تشغيل LSTM دون تحديث GeoScopeNetV2.')

## 5) إنشاء نسخة عمل محلية سريعة وضبط مسارات المشروع

In [ ]:
# كل تنفيذ يبدأ بنسخة كود حديثة من Drive، وليس نسخة Colab قديمة.
if WORK.exists(): shutil.rmtree(WORK)
shutil.copytree(PROJ, WORK, ignore=shutil.ignore_patterns('models_dl','reports','backups','*.bak','.git','__pycache__'))
os.chdir(WORK)
if str(WORK) not in sys.path: sys.path.insert(0,str(WORK))
# إلغاء نسخ src المحملة مسبقاً عند إعادة التنفيذ في الجلسة نفسها.
for mod in list(sys.modules):
    if mod=='src' or mod.startswith('src.') or mod=='preprocessing' or mod.startswith('preprocessing.'):
        del sys.modules[mod]
os.environ['LMI_FINAL_DIR'] = str(WORK/'FINAL_TRAINING_DATASET')
os.environ['LMI_MODELS_DIR'] = str(PROJ/'models_dl')
os.environ['LMI_REPORTS_DIR'] = str(PROJ/'reports')

# تاريخ المراجعة. لا يدرّب النموذج على شهر لم يكتمل.
AUDIT_DAY = AS_OF or str(datetime.now(ZoneInfo('Asia/Aden')).date())
date.fromisoformat(AUDIT_DAY)
print('WORK:',WORK,' | MODELS in Drive:',os.environ['LMI_MODELS_DIR'])
print('Audit day:',AUDIT_DAY)

## 6) فحص البيانات وتلخيص الفترات الزمنية قبل تدريب LSTM

## 6.1) إصلاح توافق إصدار سكربت LSTM قبل فحص البيانات والتدريب
إذا ظهرت رسالة `CalledProcessError ... exit status 2` فقد تكون نسخة `train_lstm.py` الموجودة في Google Drive قديمة ولا تدعم `--as-of` أو `--cpu-threads`. هذه الخلية تفحص **الواجهة الفعلية**، وتستبدل **ملفَي التدريب والسلاسل فقط عند الحاجة** بالنسخة V3 المعروفة، وتحفظ نسخة احتياطية في Drive. لا تحذف البيانات أو النماذج القديمة، ولا تخفي أخطاء التدريب.


In [ ]:
# تأكد من توافق كود LSTM الحقيقي مع خيارات دفتر Colab قبل التدريب
# إذا كانت نسخة المشروع أقدم، نُحدّث سكربتات LSTM من إصدار V3 المعروف بعد إنشاء نسخ احتياطية في Drive.
# لا تُعدل الأوزان المدربة، ولا تُحذف البيانات الأصلية، ولا تُزيف أية نتائج.
import ast, base64, zlib, hashlib, json, subprocess, sys, shutil, datetime as _dt
from pathlib import Path

_V3_FIX_PAYLOAD = {'src/forecasting/train_lstm.py': 'eNq1W1uPG0d2fp9fUWkDUbdMtmY0tuSlTQPCSt41INmGZxI4IIhGk13k9LJv292k5rIDxLKlaJUgToJFXpKXxNnospa9WtlwnLztr+C8+pfkO6eq+kJyZBnZ0Nawuy7nnDq3OudU0bKsm3v7t8TZp2efLp+c3Vs+Wv6HwJ+HeP0W/x4Jfvz67G+Wz84eCHv5dPnN8rFq/O3yO/x9iEkPl08cAvHJ2adi+R1e/xMdj3vi7C7mPVw+owHiklg+WX6J6d/o17O7y+cY8aB+fQpoHy//ixu+/+vfUNN3BOQOAQGex0TalyDs/vJbsfx8+Rztz92tLYFPdlQepInoxqLIx+4kzeXYL8owmbpl7oeJFxVlLAbd7sSPwyiUhShmYRThK5Pj0I/CY78M06QQv0hHXhmWEUZM04XMkzT3S1kMt7Zo0SDmHtb0Bah+LOy1JTs90LIDXnxMjGI2YcTH+ObxhcyB2R3NwyjwmJAjpwe2nH22/AoL/PuzfzQMf06cpb8KBjiFUffPPmY4+Lqz/Hb5L2L527N7xPe7xFoz4h/AGYiKQdC/x+L7e/8kCB6NIjwPgAVDueVzvIO9Z3ddUH7ZIWLuYNoDgvkHXu2DnhgwC8WvxAKMCphRoi92aTZrBnrAohJty3+HfJ6JK1XX0FUruIc1PFz+fvlFRcmzs1/XAHg63n7H2vRvigk1tVjRfdX2EPCfsM48I4p3HQX+jqJYSQjsgYZ8RRyvECz/G5OekUY9W36jenmxn2H8p8vPSbEU6QoUqRpp1yNwwzBc2NCq234edCfQHKWgei3PAfURaeq9przuYPqnDpH5GuGCyB6BiO8qS6klhI67JD5M+/YSm9RTspqzj5VVPEcPppKwGcUXyz+AR0oolTKg/7PlU7IfsTf2I5kT4tcdwfZNKsq8gQ0D5NfM/0+Ehrqibg19phH32Cp/DzKjNJ2N/PGMVK8hCHEQBoFMLkW5IJcAuTb0xL517YajdPUBaxqv6dHZJ0qAV5TKoY8EqNTgUe2JlLFDAveB6CnLlDiFB2LQFxCwYvdXyy95CU+0/MgCiZQHLUE6xplALuI9P1xIABa30gXchLgGY/ensrtLZF1lsu5j8HeiKU0yrKeM4g67w0q9HoFpXxn1ZzlXjpHWUxEMQJ+wQyR3yipEYiNSYf2g+4+/M3j/+D/GSGkI4+Qh7pZlWVuTPI2F503m5TyXnifCOEvzUvhJkpbKk21tmbZ8mvl5Ic37L4o0UfMzvzyIwpGZ/AFeq1nJPM6OhF+IJDNNmZ8EaMD/WbClIJCvHadxDEHrQT99/7133v1ZR1y/tn/Nu/7uhx1x6/3rN27uqecPb3zw/of7+kUmCy9MJmlHJOntjihk6RVSBh1xOw9L6dV0EhYJnZrzytxYlnk4LgxG4+s93V7Pae4CyveaOU0XvLV1E55L0T2waJ+w4O23AjkRsT+TXpwGMrKVjndE5B/JvOiIIE+zdF6yxxcGbJnm4wM3SZhvidqYxpFfFOI9WdpJ4t5Kg3kk9ST6EBYPXAhLz8P+EE0affQp5pnMbcetxjjtbsxweWvrA6FLtm7vdIQhFkL0FMF9Q/fIL8cH3iTMi7K/n89ltZK+/hbhRC9SvC12hIwKKbbd7Q14aYLCe11NtQ1T1sceSD/QNIaJ9POKnzvOVosZ2scyLzricIUd0BUPYKpl24dtXLmEPSQ1Srsi1E4HvY7o7gwdxy1+OZfyWNrdHTVbzyIZOVryt8MkSG8X9nEPBuBC7/PcP4L0Rz0RJmVHlB5pWfVSpuYR1gY9TkfQt4UM+u+liRE3zHaPpLkIwdE0iY6EGaVnFW+KRMILKc9+STtRGkv6NS8xMPJHEIgOeVZwiRC2KUZpGgl7v7PnQHuLmSv2yaeOfTClpG1LrzXLZRCOlaMgWGC7mCcVKJAZpkFBukBQCHQahyWRYBvzC8JiDEmJ8kCGOdzpe8Ifl3M/KnjL4/Xy90cdAc5hlg/JDYYd/a9CC41LRO4nU2nH/qGtOEusdhRnGypAw4t6+LFbHPiZHECobTUB2RuYA9coSB7wkgG/rIwZQH7FsA2JLThN4ELmstXxketnmUwC+3hQii4pBs9uq+NRY9B6L7HEDLCp32mpI/SOtc4GA4PyKJN9tEyi1C93LzsD13U7vBowsxp5tGGkYr1W60lY2h+VOSRCfz5a+HiiP5V3y8kPy2CDW1Mqxw4u9hPImZ21zaO5j3zQBod5c2DVbsgacoP2E5ZmSJpR7Khg4zmM3WuBH9uxi33LB/WYaDtEXD/K1YwoLYpJNYcc696Nm2jT7vGjklZIC6T1YZytBpYyKdLc9h3WJJ80aQM7tBhGMJwO//UK7KmSfGdArOJIFxzG3qVE0BHblUZPK6p+JhNEFHiG+97MMSJCZrU+gzMyS8cHYFNDoWOVwTQ8P6wzrtBgakANdiQTLAZ8mhrE/Wk9p4wwA368ZUthjRsrIAAEyWEZqY2iCI+ltWpeI4CigYOwF4pXVwcP2x47K91jmafeNIc7bhsASRGgWJh2DOIHo6FDwqDvtZEuhZ68NzhrGIpSZivNWPCrCraLaCK2HXGRVzhyGpylqKIx73ZYHhidMhS3l76ImgQviNqFYxDQzklCWDhq26Sdw0+q+aQ5xt5PlKStHinAqwJ7tqXy1LiQaATxl0QlUAtU6o5FdFqTC3yg5y3WUfigHdm9siIopcK89S4iqOlar1JtdJ/MANwdR1BnW5nHrCMWpB+xy2M82i9stVbY2WkFipa6ghYIwfudVZ9M7W/3SV8QAURHwJ1mGYIzD4FoKJMxlGfd+45y6c+Ug3Hh0gKvQU29gpbnjDt65cRx7fnyeaLjPVt9wVX6pQ84sHz4IuW1CvWcy0z5wQ4iOS+dNLfxNZ8IcOBfM6LcgEGB4b/a5yk0bjxDvw1HJ5Oy0BGZPATdXjrjVzUc3ANQF9uVbdGC54XliD/rCyudWb2mQljFfDIJx8TOI4vEh2ltpqJh0Bo1dMvUGxcLW9ME1Wv2u+iyOgAVyMP+O9jficJknAaQXN+al5PuG90inFq1XuaSzPrEUmyA1hp+KB2XAZo0IAvSRZxPYyzOPRupk0pGdRWCEjmdw39CWaSqqJwYplxQTLngnDpWZ1WHLl5k7W5pNU3Uqkxsm/FeYKmwD6u1kNKUcMR4Um3eOFUJYt2J2KT0I8s5XUNojSmDyUMf6yLUNwezocKu0MRk6BUCemtBp16oAwVn5j1LOY0pPPhdNRV4axusM6amEHWk5lI7AEEsjT0kR7xqT6zBiRLO6VB8/6//vFonAvmYNbigpHRheNqSMhsb+rlpDyJXqsWZljXc+v9Xtn2MJBmST4az3KMgAaZj7yEpjeZxotebeCW7g8SDIxXsgKjBiEDFJOxjdYOZxRWWvtiHc1Uw+AEjVRBC+yBxEQKivduURwggRbK7nQpGV+w6OqyoIlIVQxQTSuwkSKYE10go92+jf8+dUKkJTpe+Et+mPEyPc5HXItKzOcxzatV7RcRhUVBBI0yQM/RUppFhtMasqgNvUgbQVWngtvEwFBM3aQIVjusTAbUHwXKws/0l0XAjzxHcWO8mnKlgdqLmCYgEPhKsnkif6hNCkayF9gpipmauURyB6bIMxzoep1B9XqCDkwNiH1ZzqS4puU1yq9i9p1k9dP3kyPbhQfvbL0H8XhpLoWsDB/4Ce3baSMs0ck2Ypj+eQzRgTQNlLP3E1ilyEax1F2WAXtrmsUOLF3xe0aU7VedrFvgY9DFFsqQaXRDhwHKKQIe7JoSl5LhKXSmH4tBUE9KpVqaj5Cr83zCtmmNU+FWl+atATMxT5o7oI8IUEJoKgnzV8GLVqUy/XMuGa/k3hF/rlqZXC2WahwGnl1VgrfIP9rc7VzoCmVA7j4xy7tuVXdgpBLO7EukhiKCQSYf7L504tWAQVVWGp6DVw2MFXLuFhY5YRtSJPxyOVOjJwxCwjpjJo37kx6PAF9OemA62h8aqyD/xacKfsnb6FPB+3TgE0scTqqJqki1JbJFVjr9Zk2oF2td2RCtsRuAvjr4z0n6VWraSOaB3HJfLlRqOPy69yTyK2IMq1yNe9vPK2nlCzQybahz99aMEPmABux+qCi8fXZiStnbnYEdbO1W+3xGc/R2HGVVVoE7ZcWPFJHviWGEcCtcQanMC0DqdUDFBTxRljr2E989BSdmURTskOnijHBTYmyxVp0Gb4ZSqTqCHq589UHURvgWDIbR4TpO2NnLLSqiIjgkVqYPuzpADJKqpe/5i6u22u3d72l06OpHJcsomA/c6YuV3KN23aWGmzxRkaUzOJT/kKGBXlN6WeV+73HEaZzqFWS0G25illtshaLOh04zDGHbHLGOVbp3jIOJS2KlqgU24mMNRlf2BYePQcad5Os9GR7bitUNSQRZQ8ay2V6xzjzcb+2QG3lpe7BP7sOkyT6iQMyrsqSYZfn5KJP8omp1aVmEyjuaB9Ji8QoVTjouATZYea4htfMcPH3b0Vs84zMFHHa8v/06cc9LBWH6uIq9KQgdpHh4jMFUafUzmfYy4rTLjybysbebFvoH9fV3L+PmKKz+siiXaZxwfD7rRCJq436ijmYiqUTtre/PksKxc0GHb5ZgPrwIAFkhHxjN7cAwfiGkr1QyszBguwSRbY0PT61ZRKK29acwwLAw6oCjCWV/wsMGyljGBGDaYqkaIMQ5HBZzTmlBbI+0IHTj32V2YrHpM+zN509IPI/uKoy1Yk4tn2iGxqEZziYQ2WCXmpOWMOPsj0N6V2KOp6FANJsaF3oJHr5MXaxg2D2S0q+NOG7gH1viAmMO+w+YmtwUE1qVa20Q4FbQmLCTpksvlDA7MvH0gc2lruAqTeJsKbNuvO+LPRbvj0kZMTU9GkaHDAHa2abl5SDG8dY7fbX3OIeYt0f0/UsMQiJwt8fIfJP1RxKQLKlSMImk5lZfh4+mq/KEqLi9bAVEGXCBGt1Xk0KxKNUs4lFHyRpGZiH0lO66HFeqgXOfHSjuLhnpqXYvnSFo5sXR4VVS/KIJmG8JDj6sYqsIhtAlT7mVppRxPpucURXwsDBjHlCpxIUQd4yGqODkYnZojx/7JzcGFupKOXNzhwEOfp+04xPEqBe1R6NWUm6VCT1Wj7XGcaUWYSPmNR1dcaAoFnlVxvteq1KND1aa9fE4LpRifw1d0cBlO1yg95ilxSEW0LRp0eaWnygL6bVhXW3qU0Vs6YPSqxioXsRKvkab3TEBpqeS8NWENt4KpDrYwpOVbtylQajbUefvO0GlHP1adjJwDzeR958DUcfBG2LyMF0I101ehdxmYMNUXJSDlxNsoKI0g2CdaI0gZSBdymtOuNZ/yPrPo0AB0eZQoggKCoFCxnFWd1JT2PJ80J0lv2yt4ZbKgarc+82er4dVSqu8F85z00CTcVX2wzRx1XJfB7MZkQNb7rUNUlj4aigOfCpBcQvALdYJOZ67m9PRNVRkRrds8ZKkdwfUQU0Dpb7fdr6WrGV4hI70XgIZ9IrnLRRZdsxOTXP5yTsUsPmg0lb03BV8rox17xjWFauIPofHG8zKdTIzZKEU2o3Tfih5xodlMUC9swYccEAYQAcXMkSylZxIHHnr+gE2KWt/JYJ+qRCK7dBgjfD7xn3MlKsvlIkznRS2r/Rt7+8ZJ+oUSSNHhig52pXle0N2cK10eUcX01umLvfmYClBT483hb3US0YxA2GVtKEEazfNoQJpvqEOajGTTZLba+kT9xxUxoZ4bYFKdV9my9m2+/HFgKR7joW6ChWPr0Sli1WnCnA3I9bUiw3iN+HxcKp5AZM3bACIy2wRfdPKBKIOn6fxjE/MxYhXbSy5z70Xr0AwkV/CDa8iitLT3KEvsEHs6Jggw27QGa0LiHz7hUJHMxYtQRO3r0EhW1bYkClW8Eay58Dil0ykh3eOwabTJgAcWdQzXLjJsGvOWblU5oml2zkUc+7t/SrStjPQc7FrFdBJYb91VImIBHbuUFWWkCozOVlXR6j6XZ55X9Suko3/buui52W/8wBnJxvOR34iTels9bd7g/JW6Z3nr2o3+CTPhArHmwnBwAYtHrIYBfOHQ9LJkWt23ru2azib/6jFW64yTjl3UyeZLKG776DL2S5oThSNVh6teXRiKbV2bGpNYG+9mR/TEVwCjUoflmaqOrNVAuIShU8KWb1jzCHxTC1klm7zyTyFsxj9kKwMmt5iPCHHBlxyAEtHDDm3XUwpn+/YO0tnL7mv6hJ/6KTpSt7ya3ptvfRwSh0zZjXDQxbBtbI80ry4cHJoEPHTquxI62qqLcP6hy/wH0L1BWA51Rsp5dJr3rVd2d3ehVtHt/o77WkdFIH1TOWqcxjEPB2AjcY/q5oC1hiRz2Tggaq6EwUbSrlWjCq5cvnr5DTTERX+3wsV6aZP7cRoIN4BklQTMbhPm1Qn9p1dQwVR+5YXQmjoMoL0GzJ3J1auj1wxXLldQb13r7m4AyklWtZs5rOQtVltFi+QJCN6Wq2yo3Iet/MkKM6gkxvGZzSKntGaCpbB6/aQ1sgzHM48vJhXqmMk6tDSWXN+97f9kW7fw/Cv1fA6N+nw6uKMu3Bw64tIlsfPGGjWHhKiwDwe9Hs2i4s8KJ9BzWVeWWI23SY3dSE6psFRRfxVhFVjT1/UMipDKcHpQUh5JdzbrZsqt8U3ekTyjUlJvUZjQBptoxol9kIX9ncvbZt8s6doIPAfmOtXFWbqwpIzJpxWbS8jutXw6p0D5A3rLTcU+c/0AqYPus636dxpAl6AdbL5ImOXEn0dUfFU/3+AqQ/sXHNRU/4iD3pq/4zBXzjZgNJc1MIXv0NF16Bqjuc987nTOhvRUvgpqZpqbxTzgfPR+0U0nBkBBp0wGgLpidiCjrG/9FT7dW7e6168Lfx7Aj9EN0TeFjtvhNZD48DFscWBSIutclONs3i0PcnjfYiPllw3Wm2EMVD/94C+EHs4eETtongbzcThCWoN4FW2xT7d4NEISe+ay2AmtuZzH17OJ7DJEXKr3GHrffL2Gl+kF6pYSfbkEICxSRlfavqtu1dBJpH5Wt694bJkGvinS8gCs2TOLeLt5OqnrS7A8KrLoIXZrgj7g1ffU0Vef+PlRpGPC02qnmdCO4btGk2tUavBgQqXE5r0kbHlufWWovjdvKloWHiatS/TUU2dh3K/iupPJKcXebvMSU81JvRC6QflCYApWMY9jHykRh0hqJlowlxr4TlZhFxOXWIRc5bC0V4Jsy2HhYAiX9ejGDQvo5NQcuT1Zfk2naq3fUrXOLfXPIO4un5ofAj2iYG/5uCLHnWe0MpsOgU5mfJmN7gXNZvrwRJ+emPC8vnlU10xMlN5p12E658ToP6osel7E7Zy2biUpzaju2K2VLwuIn1ZLfjakXw5Qbud5FDBYnkde1/P0fTDlgrf+F+mehUE=', 'src/forecasting/series.py': 'eNqVWFuPFMcVfp9fUelISXcyNKDkacxYwsYkRHhB2EoeRq1WbXfNbGW7u5ru6r2YIBkDDkGR/JI/EKEEDHIIISh2/Mav6HnNL8l3qrq657IseGGZqdupc/nOd07heV77ZPll+7j9K2sfL++1L/H7mP61w38t7y+/XD5sv7bDfy//2L4wQ5q3c39rv6MzRshTM7W8YzY9p233lp+zX7L2Oyz+3Yh+OrF77kPU4/YFzbR/tpKetv9oX+HPMLW8D1XuQ1Yv+kH77fLh8g7pMOz5Bp932m9pVzgamckvsOs+FJgyrL5y+p+qq0/CjZ53zkKD52bhCXSHBveWD4NOyeVDhtV/Lu+2L+wM+e85I50Ypu8t7674KmT2k7VPNi6/YHd9g6Pmy6P2OTz9Jyi8w3dG/utn7X9x/Qs4Gk56BdWev/6e/e/zvzCSwWDdV+RUUo1Blf+QHgz7jDlBaG27i+FjG72ny68wgCuYmUd4X3bRRpC+gA2YeWRN6wwPR57njeaVylkczxvdVCKOmcxLVWnGi0JprqUq6pHdU3K9l8ldt+E6hnYh5VpomQu3QuPRqBsUTV4eM16zonRTJS9STOBvmXay6yoJE5XnqnBCPry2c/nKr8bs0sVPL8aXrtwYja7Cc3Z25mW1zr1odPXiB9duYPqW95trH3hj5l3Z+fSjGzuf/PrKde/2aDRKxZxliqdxU8ibjfChGY9TWU2M9jjopI+hTazmE6M7+wPbUYWgQOEjYGfeh6LhJZy9XPFcTEYMPw2WMVsJSE/qA58E9vIDdpZ59s5YlWQPBlqKOsRW6Jnq41JMa12N2b4QZQw9eZPpuODTyzyrRdBf0cyacJBwHNPBUNay8I3tQQS3lce+PdC7UhZaLCoTvTDhxeBVM4jrUiS1vWLm0UDyTH5mg+1FdGu4MRvmvPRXDges//kxe/3MZmj7EjnwNUA8xdSQuGvTF5NENYWWxeL19yN7fEcciIqJA5415HwoywvWFHNZyHpPpAygQKxURSuVSFSVAjiq1ljic42jek8w3qTSIi80Yk04caP9xGGzpFXKnbNSG0CtYodfvwlJLnQzU2MmqkpV9dRLlKgS4dlzGa91DKyWmYAKsdXOiLouKqlS31w5RjDEzan3sQf8sPPO2Wa3cXEappouL+0h2mg2VQJ5CPtn0MbsDpGHBfeDgP2E+eaUweiFzjYz3W2lyZPUC6IuF7TUmYgXlWrK2m8ma7AeM63KCQN08K3isohFkU5XM+ATqCrqyQC1shJlpRJRA4+LMBVpUzqgFarKY3Od2a4NqH6vdu2cgdOwJeiQcIMX+5DE8qbWbFcwMqOhOB9KvacazYARmRpEntUCe4x1tY24nA9qM1mb3LW6GrdyWQv2W2BMfERR9b1VX8DrNxtZCZCSFUJaJI1WcwhVZOeBKDSzHMkywff5wuFBV7BNz1Zj0OthgqNDJHqYicIP2PtT9gtEg85R4tPJKiTki9jkRe0H4R4oxUcsAqRxKo5WUaHDwz1RCYg0FEAiArAeNLHxnfNcZsdxporFdnjt4oQZ2llnwjdFfIPzUC58iqFMx9b1Yya1yANTtVyVfICMNzXv0fIB/XO3fUZFrqtJfYkMqfZ0cbOasemUefW+zLLaGwJXv5VnuyP0lZSzwx9EtO6eelYDxQl8C6ohbXiSiBIA9KLZzLOWU5UxV9D3KArTSkFsU2YyQVrW/iCxz+TVox0BRGEuqoXw6zFYberWA5hZwNt+orImL+rpreGqCfPI197t4CSvbXD44L4jU0O2NRhvH4JSHJm8KHy6aLpdAwjGNQzVvveeFwShOCozlQrfKrZl+NHsKKSVcA4TwGCA6WomsHPRyXoZaVF0gpk9f7yLhRvmrFPfmF2decgxJy5ayYDgjZawH0EJ7yTNFgo1DIRGEPjhujXhyvl3un2Lzqwujgd2G5mlsZ3zT8n7tQ7oZ29rglKZ6J4IPgQidNUkHQnDD/UeJwYlrrbOPKMKTBvVRSYXclcCO+gFC1fTXZkqUEHQ1pPkK7S7snVgTM3jLgo8KngO/iUQjs3xhNwFCma4cagSSFpzL6N7bVH42BSITiel2Fwcsq7GE+nntDuXpn69R33EsZFYmI4k56mgcqiY5shV3dUZx1tv7zCofzup++z8PLX126GpCUVe6uPJZvhveTaAYAD7hVIXnXlTEycUyoSUoGXEeSbOtgcJblusoJYZDrWtRgzvLkTfXeSoI6gh/ZAf0bBvXmyJw0sgMxXc5M/usd/hGAktPxM+8ZapVX5FUaN8j01Rm56zAtQu0asVgxKI3MO1sQsE9UBxlxhdJ/A7iFOHa7BpaoALgkR1gFh3Z3skACsKjxa8LSotkybjFfW6qTRQngtOhdvGT0ETiLFV13bBh0CBxbkLhgp5gTAOwaC2CnsAB78ouyqsgtm5yHpsSNo5kIDwoAWCUOtqOjtmZ87T74pId9XMbI8MsLvBxFz3c3ae6gTvGofOa85iL1oXNRjiGxljo3MPL1ra7oreEWK9223D1RNZHZuYYheROoIfnIJDmRIdYtMMusBzE/o4HxkzzYYituk7NdKwnVpn4mm0enF3c2SnELx+xlnYH7rgXNWfQUyc8AvGkZlS+7s82V914js5QxZ1M5/LRFI/aCWe6hNS6BSfDB3rlPwzc1qecT7JKHfX2rpxr9bAJ0P3NlSxrc5ktQzZGmvBkRSE7GzRJ/dssxb3ad6A8+E1fy3D++SHASclv6kLuAH3hKk88I17Vg+RhDLjifDPIY3LsOAFfMaPZE3Su1xDxq4e6dc7wriEMlIh5oIJnuyx2jxVfrpGIPA9Chgqknm70KOxb/UPDd1YfqjUIaqMPZiR3jM0BzMbDUpuaTLbGBV2fdqAISQnxQusr32zZSZ1FMpMJbNJF9vIPei61B6oAzWuoxi46qSDdZOv7l8oRbihO0+iB1toIXNYdFW0SxpnbsjLEpDxb9l44xEI9vAslDp0T1wC9Qv9RRPIaHAaE2O8bsZbrGR+nDinwYRUgzBq7CnXdpXKfLIouD1YiKymqXWic6FxWsuO5CgxbaHrny0+WQf04kVqMVn7m3pQMU4gBjDongR0K53DReix1sS5vnxmHdWrH6293U9nELXv0Q09wMwta7xCci0isd8nyiBqOBcB9G5w5nwUbHja6yO1wjtWzHasLEI3k8ohcjuMnk0oYjCDaqc9UsOzD1csEWpXFxwPbif8pnBnekI+o9EKMJzEHnhdNY/t+7wj1YH3NmRvMy8EiaMka1KRxrJwPWjXgNiNb/7PHPD2/wG1+1E9'}
_V3_FIX_SHA = {'src/forecasting/train_lstm.py': 'd3afbd9f2d30e8e264e78e0e2dd31784e544e9bcb80a8ad217d27474b3d1a4eb', 'src/forecasting/series.py': '7d81881efed309d23b372d9f89fd1308f653ba30c7b30a35cd0335ea6e4d8358'}

def _has_supported_lstm_api():
    trainer=(WORK/'src/forecasting/train_lstm.py').read_text(encoding='utf-8')
    series=(WORK/'src/forecasting/series.py').read_text(encoding='utf-8')
    t_ast=ast.parse(trainer)
    s_ast=ast.parse(series)
    tr_func=next((n for n in t_ast.body if isinstance(n, ast.FunctionDef) and n.name=='run_family'), None)
    se_func=next((n for n in s_ast.body if isinstance(n, ast.FunctionDef) and n.name=='build_family'), None)
    if not tr_func or not se_func:
        return False
    tr_args={a.arg for a in tr_func.args.args+tr_func.args.kwonlyargs}
    se_args={a.arg for a in se_func.args.args+se_func.args.kwonlyargs}
    return ('--as-of' in trainer and '--cpu-threads' in trainer
            and {'as_of'}.issubset(tr_args) and {'as_of'}.issubset(se_args))

if not _has_supported_lstm_api():
    stamp=_dt.datetime.now().strftime('%Y%m%d_%H%M%S')
    backup_root=PROJ/'backups'/('lstm_source_before_cli_fix_'+stamp)
    audit={}
    for rel,compressed in _V3_FIX_PAYLOAD.items():
        original=PROJ/rel
        local=WORK/rel
        if not original.is_file() or not local.is_file():
            raise FileNotFoundError(f'ملف المشروع ناقص: {rel}')
        revised=zlib.decompress(base64.b64decode(compressed))
        expected=_V3_FIX_SHA[rel]
        if hashlib.sha256(revised).hexdigest()!=expected:
            raise RuntimeError(f'فشل التحقق من سلامة نسخة التدريب: {rel}')
        ast.parse(revised.decode('utf-8'))
        b=backup_root/rel
        b.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(original,b)
        audit[rel]={'original_sha256':hashlib.sha256(original.read_bytes()).hexdigest(),
                    'new_sha256':expected,'backup':str(b)}
    # النسخة المحلية/Drive تتلقى الإصلاح نفسه لتجنب تكرار الخطأ في الجلسات المقبلة.
    for rel,compressed in _V3_FIX_PAYLOAD.items():
        revised=zlib.decompress(base64.b64decode(compressed))
        for root in (PROJ,WORK):
            dst=root/rel
            tmp=dst.with_name(dst.name+'.new_lmi_cli_fix')
            tmp.write_bytes(revised)
            tmp.replace(dst)
    backup_root.mkdir(parents=True,exist_ok=True)
    (backup_root/'fix_manifest.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
    print('✓ تم تحديث ملفَي تدريب LSTM وتهيئة السلاسل إلى إصدار V3 المتوافق.')
    print('✓ نسخ احتياطية محفوظة في:',backup_root)
else:
    print('✓ كود LSTM متوافق بالفعل مع --as-of و--cpu-threads؛ لا تعديل.')

# التحقق عبر واجهة الأوامر الفعلية (وليس مجرد وجود ملف Python).
_probe=subprocess.run([sys.executable,'-m','src.forecasting.train_lstm','--help'],
    cwd=WORK, env=os.environ.copy(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding='utf-8', errors='replace')
if _probe.returncode != 0:
    raise RuntimeError('فشل فحص سكربت LSTM قبل التدريب:\n'+_probe.stdout[-3500:])
for flag in ('--families','--data_dir','--as-of','--cpu-threads'):
    if flag not in _probe.stdout:
        raise RuntimeError(f'خيار {flag} غير مدعوم في سكربت LSTM المحلي؛ لا يبدأ التدريب.')
print('✓ تم التحقق من دعم جميع خيارات التدريب قبل التشغيل.')


In [ ]:
from src.common import CONFIG
from src.forecasting.series import build_family
from IPython.display import display

u = pd.read_csv(WORK/'FINAL_TRAINING_DATASET/unique_opportunities.csv',
                usecols=['job_id','opportunity_type','posting_date'],low_memory=False)
print('عدد الفرص في الملف:',len(u))
print('التواريخ بعد التحويل:',pd.to_datetime(u.posting_date,errors='coerce').min(),
      '→',pd.to_datetime(u.posting_date,errors='coerce').max())
print('الوظائف والتدريب العملي:',int(u.opportunity_type.isin(['JOB','INTERNSHIP']).sum()))
del u

family_check=[]
for family in LSTM_FAMILIES:
    pack=build_family(family, WORK/'FINAL_TRAINING_DATASET', as_of=date.fromisoformat(AUDIT_DAY))
    row={'family':family,'status':pack['status'],'window':str(pack.get('window')),
         'months':pack.get('months',0),'series_count':len(pack.get('ready',[]))}
    family_check.append(row)
    print(row)
    if pack['status']!='ok':
        raise RuntimeError(f'لا يوجد تدريب LSTM صالح للعائلة {family}: {pack["status"]}')
display(pd.DataFrame(family_check))
print('فحص جاهزية LSTM: ناجح — سيتم حساب السلاسل من البيانات الفعلية.')

## 7) **تدريب LSTM الحقيقي** (أهم خلية)

- يشغّل `python -m src.forecasting.train_lstm --families ...` لكل عائلة على حدة.
- يستخدم **السلاسل الشهرية** ونافذة 6 أشهر وتقسيماً زمنياً Train/Validation/Test من كود المشروع.
- يجرّب إعدادات Hidden Size / Learning Rate ويستخدم Early Stopping وفق إعدادات المشروع، ثم يقيس MAE وRMSE على Test.
- يكتب أولاً في **مجلد مرحلي محلي** ثم يفحص الأوزان والنتائج قبل نسخها إلى Drive؛ تحتفظ النسخ السابقة بنسخة احتياطية.
- إذا انقطعت جلسة Colab، فإن `LSTM_TRAIN_MODE='missing'` يسمح باستعمال العائلات السليمة المحفوظة، أما `force` فيعيد تدريب الجميع.

> الأرقام لا تُخمن أو تُثبّت يدوياً في الدفتر. وتستخدم المقارنة طريقة **one-step-ahead**؛ توقعات الأشهر الستة القادمة مختلفة ولا تُعامل كأنها Test.

In [ ]:
import math, time, json, shutil, subprocess, os
from pathlib import Path
from src.forecasting.train_lstm import make_model

MODELS_DRIVE=PROJ/'models_dl/lstm'
EVAL_DRIVE=PROJ/'reports/evaluation'
STAGE=Path('/content/lmi_lstm_stage')
BACKUP=PROJ/'backups'/('lstm_'+datetime.now().strftime('%Y%m%d_%H%M%S'))

def verify_family(family, model_dir, report_dir):
    model_dir,report_dir=Path(model_dir),Path(report_dir)
    req=[model_dir/'lstm.pt',model_dir/'config.json',model_dir/'scaler.json',
         report_dir/'metrics.json',report_dir/'test_predictions.csv',
         report_dir/'training_history.csv',report_dir/'future_forecast.csv']
    absent=[str(p) for p in req if not p.is_file() or p.stat().st_size==0]
    if absent: raise RuntimeError(f'{family}: ملفات التدريب أو التقييم ناقصة: {absent}')
    cfg=json.loads((model_dir/'config.json').read_text(encoding='utf8'))
    met=json.loads((report_dir/'metrics.json').read_text(encoding='utf8'))
    if cfg['family']!=family or not met.get('trained') or met.get('family')!=family:
        raise RuntimeError(f'{family}: البيانات الوصفية متعارضة أو التدريب غير مكتمل')
    if not (cfg['train_period'][-1]<cfg['validation_period'][0] <= cfg['validation_period'][-1] < cfg['test_period'][0]):
        raise RuntimeError(f'{family}: ترتيب الفترات الزمنية غير صحيح')
    net=make_model(int(cfg['hidden_size']),int(CONFIG['lstm']['num_layers']),float(cfg['dropout']))
    state=torch.load(model_dir/'lstm.pt',map_location='cpu',weights_only=True)
    net.load_state_dict(state,strict=True)
    net.eval()
    with torch.no_grad():
        check=net(torch.zeros(2,int(cfg['lookback']),1)).numpy()
    if check.shape!=(2,) or not np.isfinite(check).all():
        raise RuntimeError('تحميل أوزان LSTM أو الاستدلال فشل')
    preds=pd.read_csv(report_dir/'test_predictions.csv')
    required={'month','item','actual','lstm','naive','moving_avg_3'}
    if preds.empty or not required.issubset(preds.columns):
        raise RuntimeError(f'{family}: ملف تنبؤات الاختبار غير صالح')
    if not (preds.month.min()>=cfg['test_period'][0] and preds.month.max()<=cfg['test_period'][-1]):
        raise RuntimeError(f'{family}: فترات Test في CSV لا تطابق الإعدادات')
    for key in ('lstm','naive','moving_avg_3'):
        arr=preds[['actual',key]].apply(pd.to_numeric,errors='coerce').dropna()
        if arr.empty: raise RuntimeError(f'{family}: لا توجد قيم فعلية لتقييم {key}')
        mae=float((arr.actual-arr[key]).abs().mean())
        rmse=float(np.sqrt(np.mean((arr.actual-arr[key])**2)))
        stored=met['test'][key]
        if abs(mae-float(stored['mae']))>1e-5 or abs(rmse-float(stored['rmse']))>1e-5:
            raise RuntimeError(f'{family}/{key}: المقاييس المحفوظة لا تطابق تنبؤات Test')
    return met

def promote_dir(src,dst,backup):
    """نسخ المخرجات قبل الاستبدال؛ ولا يُمس الأصل إلا بعد التأكد من سلامة النسخة المرحلية."""
    src,dst,backup=map(Path,(src,dst,backup))
    dst.parent.mkdir(parents=True,exist_ok=True)
    tmp=dst.parent/(dst.name+'_pending_'+str(os.getpid()))
    if tmp.exists():shutil.rmtree(tmp)
    shutil.copytree(src,tmp)
    try:
        if dst.exists():
            backup.parent.mkdir(parents=True,exist_ok=True)
            if backup.exists():raise FileExistsError(backup)
            shutil.copytree(dst,backup)
            shutil.rmtree(dst)
        tmp.rename(dst)
    except Exception:
        if not dst.exists() and backup.is_dir():shutil.copytree(backup,dst)
        raise
    finally:
        if tmp.exists():shutil.rmtree(tmp)

MODELS_DRIVE.mkdir(parents=True,exist_ok=True)
EVAL_DRIVE.mkdir(parents=True,exist_ok=True)
status={}
for family in LSTM_FAMILIES:
    target_model=MODELS_DRIVE/family
    target_eval=EVAL_DRIVE/('lstm_'+family)
    if LSTM_TRAIN_MODE=='missing':
        try:
            verify_family(family,target_model,target_eval)
            status[family]='reused_verified'
            print('✓ موجود وسليم، تخطيت إعادة التدريب:',family)
            continue
        except (RuntimeError,KeyError,ValueError,FileNotFoundError,AssertionError) as e:
            print(f'{family}: سأعيد التدريب لأن الحالة ليست سليمة: {e}')
    if STAGE.exists():shutil.rmtree(STAGE)
    env=os.environ.copy()
    env['LMI_MODELS_DIR']=str(STAGE/'models_dl')
    env['LMI_REPORTS_DIR']=str(STAGE/'reports')
    cmd=[sys.executable,'-m','src.forecasting.train_lstm','--families',family,
         '--data_dir',str(WORK/'FINAL_TRAINING_DATASET'),
         '--as-of',AUDIT_DAY,'--cpu-threads','2']
    t0=time.monotonic()
    print('\n=== بدء تدريب LSTM:',family,'===',flush=True)
    # بثّ جميع رسائل التدريب مع حفظها في ملف Log داخل Drive.
    # الأوامر/الوسائط تؤكّدها خلية الفحص المسبق، ولا نُخفي stderr.
    logs=PROJ/'reports'/'colab_logs'
    logs.mkdir(parents=True,exist_ok=True)
    logfile=logs/f'lstm_{family}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
    with logfile.open('w',encoding='utf-8') as lf:
        proc=subprocess.Popen(cmd,cwd=WORK,env=env,stdout=subprocess.PIPE,
              stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        for line in proc.stdout:
            print(line,end='',flush=True)
            lf.write(line)
        returncode=proc.wait()
    if returncode:
        tail='\n'.join(logfile.read_text(encoding='utf-8').splitlines()[-35:])
        raise RuntimeError(f'فشل تدريب {family} (exit code={returncode}). راجع {logfile}\n{tail}')
    model_src=STAGE/'models_dl/lstm'/family
    eval_src=STAGE/'reports/evaluation'/('lstm_'+family)
    met=verify_family(family,model_src,eval_src)
    promote_dir(model_src,target_model,BACKUP/'models_dl/lstm'/family)
    promote_dir(eval_src,target_eval,BACKUP/'reports/evaluation'/('lstm_'+family))
    status[family]='newly_trained_and_verified'
    summ_path=EVAL_DRIVE/'lstm_summary.json'
    summ=json.loads(summ_path.read_text(encoding='utf8')) if summ_path.is_file() else {}
    summ[family]={k:met.get(k) for k in ('trained','series','test','test_period','lstm_beats_naive_mae','lstm_beats_ma3_mae')}
    temporary=summ_path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(summ,ensure_ascii=False,indent=2),encoding='utf8')
    temporary.replace(summ_path)
    print(f'✓ {family}: test MAE={met["test"]["lstm"]["mae"]:.6f} | الوقت {(time.monotonic()-t0)/60:.1f} دقيقة | محفوظ في Drive')

print('\nحالة تدريب LSTM:',status)

## 8) **تقييم LSTM** من نتائج Test الفعلية ورسم المقارنة مع خطوط الأساس

In [ ]:
import matplotlib.pyplot as plt
rows=[]
for fam in LSTM_FAMILIES:
    met=verify_family(fam,MODELS_DRIVE/fam,EVAL_DRIVE/('lstm_'+fam))
    t=met['test']
    options={key:t[key]['mae'] for key in ('lstm','naive','moving_avg_3')}
    rows.append({'Family':fam,'Series':met['series'],
                 'Test period':' → '.join(met['test_period']),
                 'Test n':t['lstm']['n'],
                 'LSTM MAE':t['lstm']['mae'], 'LSTM RMSE':t['lstm']['rmse'],
                 'Naive MAE':t['naive']['mae'],'Moving avg 3 MAE':t['moving_avg_3']['mae'],
                 'Best method':min(options,key=options.get)})
df_results=pd.DataFrame(rows)
display(df_results)
fig,ax=plt.subplots(figsize=(11,4.5))
x=np.arange(len(df_results)); width=.25
for offset,col,label in [(-1,'LSTM MAE','LSTM'),(0,'Naive MAE','Naive'),(1,'Moving avg 3 MAE','Moving Avg 3')]:
    ax.bar(x+offset*width,df_results[col],width=width,label=label)
ax.set_xticks(x,df_results['Family']);ax.set_ylabel('MAE on held-out test (lower is better)')
ax.set_title('LMI • LSTM versus statistical baselines');ax.legend();ax.grid(axis='y',alpha=.2)
plt.tight_layout()
outcsv=EVAL_DRIVE/'lstm_unified_notebook_results.csv'
outpng=EVAL_DRIVE/'lstm_unified_notebook_comparison.png'
df_results.to_csv(outcsv,index=False,encoding='utf-8-sig')
fig.savefig(outpng,dpi=165)
plt.show()
print('حُفظ جدول الاختبار والرسم في:',EVAL_DRIVE)

## 9) تدريب **AraBERT** للتصنيف وNER (عند توفر GPU أو اختيار CPU يدوياً)

تُستخدم 3 مهام:
1. `opportunity_type`: تصنيف نوع الفرصة.
2. `geographic_scope`: النطاق الجغرافي التفصيلي.
3. `ner`: استخراج الكيانات بوسوم BIO.

**التدريب من البداية باستخدام `aubmindlab/bert-base-arabertv02` مع Fine-Tuning.** لا يتم اختراع Gold؛ التقييم النهائي على Silver Test. عند تفعيل `APPLY_GEO_PATCH` يظل تطبيق Streamlit يفضل النموذج الجغرافي المبسط GeoScopeNetV2 (3 فئات) مهما كانت أوزان AraBERT الجغرافي محفوظة؛ هاتان مهمتان مختلفتان.

### 9.1) إصلاح تنسيق AMP وScheduler قبل بدء AraBERT — محلياً وبطريقة قابلة للفحص

In [ ]:
import ast,re
MARK='AMP_SAFE_SCHEDULER_STEP_V2'

def repair_amp(path):
    path=Path(path)
    text=path.read_text(encoding='utf8')
    if MARK in text:return 'already fixed'
    p=re.compile(r'(?P<indent>^[ \t]*)scaler\.step\(opt\)\s*\n[ \t]*scaler\.update\(\)\s*\n[ \t]*opt\.zero_grad\((?:set_to_none=True)?\)\s*\n[ \t]*sch\.step\(\)', re.M)
    matches=list(p.finditer(text))
    if len(matches)!=1:
        raise RuntimeError(f'{path.name}: لا يوجد موضع واحد معروف لـAMP/Scheduler؛ لا أعدّل الكود عشوائياً ({len(matches)})')
    ind=matches[0].group('indent')
    new='\n'.join([f'{ind}# {MARK}',f'{ind}amp_scale_before = scaler.get_scale()',
                   f'{ind}scaler.step(opt)',f'{ind}scaler.update()',
                   f'{ind}amp_skipped_step = bool(amp and scaler.get_scale() < amp_scale_before)',
                   f'{ind}opt.zero_grad(set_to_none=True)',f'{ind}if not amp_skipped_step:',
                   f'{ind}    sch.step()'])
    updated=text[:matches[0].start()]+new+text[matches[0].end():]
    ast.parse(updated,filename=str(path))
    path.with_suffix('.py.pre_amp_backup').write_text(text,encoding='utf8')
    path.write_text(updated,encoding='utf8')
    return 'patched'

if TRAIN_ARABERT:
    for name in ('train_classifier.py','train_ner.py'):
        print(name,repair_amp(WORK/'src/training'/name))
else:
    print('لا حاجة لتعديل سكربت AraBERT حالياً؛ LSTM يعمل مستقلاً.')

### 9.2) التحقق من نماذج AraBERT الموجودة، ثم تدريب الناقص أو إعادة التدريب حسب الخيارات

In [ ]:
ARABERT_TASKS=('opportunity_type','geographic_scope','ner')
HF_MODELS=PROJ/'models_dl'
HF_EVAL=EVAL_DRIVE

def hf_complete(folder):
    folder=Path(folder)
    weights=any((folder/name).is_file() and (folder/name).stat().st_size>0 for name in
                ('model.safetensors','pytorch_model.bin','model.safetensors.index.json','pytorch_model.bin.index.json'))
    return weights and all((folder/name).is_file() for name in ('config.json','training_config.json','tokenizer_config.json'))

if not TRAIN_ARABERT:
    print('AraBERT: تخطيت التدريب وفق الإعدادات. LSTM أعلاه تم تدريبه/تقييمه بصورة مستقلة.')
else:
    from src.data.loaders import load_classification,load_ner
    for task in ARABERT_TASKS:
        dest=(HF_MODELS/'ner') if task=='ner' else (HF_MODELS/'classification'/task)
        if hf_complete(dest) and not RETRAIN_ARABERT:
            print('✓ نموذج AraBERT محفوظ (ملفات رئيسية موجودة):',task)
            continue
        if task=='ner':
            recs,labels=load_ner(WORK/'FINAL_TRAINING_DATASET',splits=['train','validation'])
            print('NER train',len(recs['train']),'validation',len(recs['validation']),'BIO tags',len(labels))
        else:
            recs,labels=load_classification(task,WORK/'FINAL_TRAINING_DATASET',splits=['train','validation'])
            print(task,'train',len(recs['train']),'validation',len(recs['validation']))
        stage_dir=Path('/content/lmi_arabert_stage')/task
        if stage_dir.exists():shutil.rmtree(stage_dir)
        stage_dir.mkdir(parents=True)
        cmd=([sys.executable,'-m','src.training.train_ner'] if task=='ner' else
             [sys.executable,'-m','src.training.train_classifier','--task',task])
        cmd+=['--out',str(stage_dir),'--data_dir',str(WORK/'FINAL_TRAINING_DATASET')]
        print('=== Fine-tuning:',task,'===',flush=True)
        subprocess.run(cmd,check=True,cwd=WORK,env=os.environ.copy())
        if not hf_complete(stage_dir):
            raise RuntimeError(f'{task}: تم إنهاء الأمر دون إنتاج ملفات أوزان مكتملة؛ لن أصفه بأنه مدرب')
        # تحقق عملي من قابلية تحميل checkpoint.
        from transformers import AutoModelForSequenceClassification,AutoModelForTokenClassification,AutoTokenizer
        loader=AutoModelForTokenClassification if task=='ner' else AutoModelForSequenceClassification
        model=loader.from_pretrained(str(stage_dir),local_files_only=True)
        tok=AutoTokenizer.from_pretrained(str(stage_dir),local_files_only=True)
        del model,tok
        if torch.cuda.is_available():torch.cuda.empty_cache()
        promote_dir(stage_dir,dest,PROJ/'backups'/('arabert_'+datetime.now().strftime('%Y%m%d_%H%M%S'))/task)
        print('✓ نموذج جديد تم تدريبه ونجح تحميله وحُفظ في Drive:',task)

### 9.3) التقييم النهائي لنماذج AraBERT التي تتوفر أوزانها (Silver Test)

In [ ]:
if not EVALUATE_ARABERT:
    print('تم تعطيل تقييم AraBERT من الإعدادات، وليس نتيجة خطأ.')
else:
    if importlib.util.find_spec('transformers') is None:
        print('Transformers غير مثبتة. يمكن إكمال LSTM وحده وتغيير EVALUATE_ARABERT=False.')
    else:
        for task in ARABERT_TASKS:
            dest=(HF_MODELS/'ner') if task=='ner' else (HF_MODELS/'classification'/task)
            if not hf_complete(dest):
                print('لا توجد أوزان AraBERT مكتملة — لا أدّعي وجود تقييم:',task)
                continue
            out=HF_EVAL/('ner' if task=='ner' else 'classification_'+task)
            cmd=([sys.executable,'-m','src.evaluation.evaluate_ner'] if task=='ner' else
                 [sys.executable,'-m','src.evaluation.evaluate_classifier','--task',task])
            cmd+=['--model_dir',str(dest),'--data_dir',str(WORK/'FINAL_TRAINING_DATASET'),'--out',str(out)]
            print('=== Test Silver:',task,'===',flush=True)
            subprocess.run(cmd,check=True,cwd=WORK,env=os.environ.copy())
            result=out/'metrics.json'
            if not result.is_file():raise RuntimeError('لم يتم حفظ metrics.json بعد التقييم')
            print('✓ تقرير التقييم:',result)

## 10) فحص نهائي للمشروع ومراجعة الملفات التي يمكن تقديمها للجنة

نتائج الاختبار لا تصبح **Gold** لمجرد نجاح التشغيل. لا تعرض توقعات LSTM المستقبلية على أنها قيم مرصودة، ولا تحذف ملف المقارنة العلمية لمجرد إخفاء جدولها من واجهة Streamlit.

In [ ]:
# فحص الملفات الأساسية دون الحاجة إلى تشغيل Streamlit أو تحميل AraBERT.
for fam in LSTM_FAMILIES:
    verify_family(fam,MODELS_DRIVE/fam,EVAL_DRIVE/('lstm_'+fam))
print('✓ أربعة أنواع من نماذج LSTM: الأوزان والنتائج متوافقة.')
print('✓ جدول مقارنة MAE/RMSE:',EVAL_DRIVE/'lstm_unified_notebook_results.csv')
print('✓ الرسم:',EVAL_DRIVE/'lstm_unified_notebook_comparison.png')
print('✓ مجلد النماذج:',PROJ/'models_dl/lstm')
print('✓ كود تدريب LSTM الحقيقي:',WORK/'src/forecasting/train_lstm.py')
status_path=PROJ/'reports'/'unified_notebook_execution_status.json'
status_data={'generated_at':datetime.now().isoformat(timespec='seconds'),
    'as_of':AUDIT_DAY,'lstm_status':status,'lstm_result_file':str(EVAL_DRIVE/'lstm_unified_notebook_results.csv'),
    'arabert_training_enabled':bool(TRAIN_ARABERT),'label_quality':'silver for NLP unless verified human Gold',
    'notebook_training_claim':'only tasks that actually executed in this session are marked newly_trained_and_verified'}
status_path.write_text(json.dumps(status_data,ensure_ascii=False,indent=2),encoding='utf8')
print('حُفظ تقرير حالة التشغيل الفعلي:',status_path)

## 11) تشغيل تطبيق Streamlit (اختياري؛ يحتاج اتصالاً بالإنترنت لنفق Cloudflare)

In [ ]:
if not RUN_STREAMLIT:
    print('لم يُشغّل Streamlit — LSTM والتقييم اكتملوا بشكل مستقل. لتشغيل التطبيق اجعل RUN_STREAMLIT=True ثم أعد تشغيل هذه الخلية.')
else:
    import urllib.request, threading, queue, re
    pkg='streamlit'
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable,'-m','pip','install','streamlit>=1.40'],check=True)
    log_path=Path('/content/lmi_streamlit.log')
    with log_path.open('w',encoding='utf8') as out:
        app=subprocess.Popen([sys.executable,'-m','streamlit','run','app_dl/main.py',
                              '--server.port','8501','--server.headless','true'],
                             cwd=WORK,env=os.environ.copy(),stdout=out,stderr=subprocess.STDOUT)
        try:
            ok=False
            for _ in range(60):
                if app.poll() is not None:raise RuntimeError('توقف Streamlit: '+log_path.read_text(encoding='utf8')[-2500:])
                try:
                    with urllib.request.urlopen('http://127.0.0.1:8501/_stcore/health',timeout=1) as r:
                        if r.status==200:ok=True;break
                except Exception:time.sleep(1)
            if not ok:raise RuntimeError('لم يرد Streamlit؛ راجع /content/lmi_streamlit.log')
            print('✓ الواجهة تستجيب داخل Colab')
            cloud=Path('/content/cloudflared')
            if not cloud.is_file():
                urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',str(cloud))
                cloud.chmod(0o755)
            tunnel=subprocess.Popen([str(cloud),'tunnel','--url','http://127.0.0.1:8501'],
                                    stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            q=queue.Queue()
            threading.Thread(target=lambda: [q.put(line) for line in tunnel.stdout],daemon=True).start()
            public=None
            until=time.monotonic()+90
            while time.monotonic()<until:
                if tunnel.poll() is not None and q.empty():break
                try:line=q.get(timeout=1)
                except queue.Empty:continue
                mt=re.search(r'https://[\w-]+\.trycloudflare\.com',line)
                if mt:public=mt.group(0);break
            if public is None:raise RuntimeError('لم يتم إنشاء رابط Cloudflare. التطبيق المحلي يمكن أن يكون شغالاً.')
            print('رابط التطبيق المؤقت (عام — لا تستخدم بيانات شخصية):',public,flush=True)
            while app.poll() is None and tunnel.poll() is None:time.sleep(3)
        finally:
            for p in (locals().get('tunnel'),app):
                if p is not None and p.poll() is None:p.terminate()
    print('سجل الواجهة:',log_path)

## ملاحظات مهمة للتقرير الأكاديمي

- `LMI_GEO_NEURAL_V3_PATCH.zip` يحسّن التصنيف الجغرافي فقط، ولا يتضمن كامل مصادر بيانات LSTM.
- لا يمكن ضمان تفوق LSTM على Naive أو المتوسط المتحرك؛ المقاييس الفعلية تُعرض في الجدول داخل الدفتر وتُحفظ في Drive.
- إذا لم يتوفر GPU، **يُدرَّب LSTM فعلياً على CPU**، بينما AraBERT يُتخطى افتراضياً توفيراً للموارد، ويمكن تفعيله يدوياً.
- عند إعادة فتح الدفتر، تُحفظ أوزان النماذج والتقارير في Google Drive، لكن مخرجات الخلايا لن تظهر إلا بعد تنفيذها في الجلسة الجديدة أو حفظ ملف الدفتر بعد التشغيل.